# Assignment 1: Vector Database Creation and Retrieval
## Day 6 Session 2 - RAG Fundamentals

The goal is not to build a completely new system from scratch. The goal is to repeat the same flow with guided TODOs so the first notebook feels intuitive.

## What you will build

You will build a small multimodal RAG retrieval pipeline using LlamaIndex and LanceDB:

1. Mount Google Drive and install dependencies
2. Configure LlamaIndex settings
3. Explore a folder containing different file types
4. Load documents using `SimpleDirectoryReader`
5. Create a LanceDB vector store
6. Create a `StorageContext`
7. Build a `VectorStoreIndex`
8. Retrieve relevant chunks for a query
9. Inspect retrieved chunks and metadata
10. Optionally generate an answer using a query engine

## Main idea

Yesterday's session and notebooks showed the full RAG system end-to-end. This assignment asks you to complete the similar flow step by step.

**INSTRUCTIONS:**
1. Complete each function by replacing the TODO comments with actual implementation
2. Run each cell after completing the function to test it

## 0. Mount Google Drive

Use this if you are running the notebook in Google Colab.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Install dependencies

Use the same requirements file path used in class. If your folder path is different, update it below.

In [3]:
!pip install -q uv
# TODO: pass your own requirements.txt file
!uv pip install --system -r "/content/drive/MyDrive/ai-accelerator-C6-main/Day 6/session_2/requirements.txt"   # change this to your own path of requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.8/24.8 MB 49.7 MB/s eta 0:00:00
Using Python 3.12.13 environment at: /usr
Resolved 186 packages in 3.40s
Prepared 45 packages in 10.81s
Uninstalled 4 packages in 553ms
Installed 45 packages in 378ms
 + banks==2.4.2
 + colorama==0.4.6
 + dataclasses-json==0.6.7
 + deprecated==1.3.1
 + dirtyjson==1.0.8
 + filetype==1.2.0
 + griffe==2.0.2
 + griffecli==2.0.2
 + griffelib==2.0.2
 - huggingface-hub==1.11.0
 + huggingface-hub==0.34.0
 + jedi==0.20.0
 + lance-namespace==0.7.2
 + lance-namespace-urllib3-client==0.7.2
 + lancedb==0.30.2
 + llama-cloud==1.6.0
 + llama-index==0.14.16
 + llama-index-cli==0.5.5
 + llama-index-core==0.14.21
 + llama-index-embeddings-huggingface==0.7.0
 + llama-index-embeddings-openai==0.5.2
 + llama-index-indices-managed-llama-cloud==0.11.1
 + llama-index-instrumentation==0.5.0
 + llama-index-llms-huggingface-api==0.7.0
 + llama-index-llms-openai==0.6.26
 + llama-index-llms-openai-like==0.6.0
 + llama-index-llms-openrou

## 2. Set OpenRouter API key

We use OpenRouter for the LLM, similar to the class notebook. The embedding model will be local.

In [4]:
import os
from getpass import getpass

os.environ["OPENROUTER_API_KEY"] = getpass("Enter your OpenRouter key: ")
print("OpenRouter key set successfully")

Enter your OpenRouter key: ··········
OpenRouter key set successfully


## 3. Imports and configuration

Before running the full pipeline, check these values:

- `data_path`: folder that contains your input files (CHANGE IT ACCORDING TO YOUR OWN PATH)
- `vector_db_path`: where LanceDB will store vectors
- `index_storage_path`: where LlamaIndex will persist index information
- `chunk_size` and `chunk_overlap`: same concepts discussed in class

In [5]:
import os
import time
from pathlib import Path
from typing import List


CONFIG = {
    "llm_model": "gpt-5-mini",
    "embedding_model": "local:BAAI/bge-small-en-v1.5",
    "chunk_size": 512,
    "chunk_overlap": 50,
    "similarity_top_k": 5,
    "data_path": "/content/drive/MyDrive/ai-accelerator-C6-main/Day 6/session_2/data",  # change the path accordingly
    "vector_db_path": "/content/storage/assignment_multimodal_vectordb",
    "index_storage_path": "/content/storage/assignment_multimodal_index",
    "table_name": "assignment_documents"
}

os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("Configuration loaded")
print("Data path:", CONFIG["data_path"])
print("Vector DB path:", CONFIG["vector_db_path"])
print("Index storage path:", CONFIG["index_storage_path"])

Configuration loaded
Data path: /content/drive/MyDrive/ai-accelerator-C6-main/Day 6/session_2/data
Vector DB path: /content/storage/assignment_multimodal_vectordb
Index storage path: /content/storage/assignment_multimodal_index


## 4. Configure LlamaIndex settings

This step defines:

- which LLM will generate answers
- which embedding model will convert text into vectors
- how text will be chunked

### Your task

Complete the function below.

Hint: This is very similar to the `configure_llamaindex_settings()` function from the class notebook.

In [6]:
from llama_index.core import Settings
from llama_index.llms.openrouter import OpenRouter
from llama_index.core.embeddings import resolve_embed_model
from llama_index.core.node_parser import SentenceSplitter

def configure_llamaindex_settings():

    """Configure LlamaIndex global settings using hardcoded configuration."""

    # Set up LLM with OpenRouter using hardcoded model
    Settings.llm = OpenRouter(
        api_key=os.getenv("OPENROUTER_API_KEY"),
        model=CONFIG["llm_model"]
    )
    print(f"✓ LLM configured: {CONFIG['llm_model']}")

    # Set up local embedding model (downloads locally first time, then cached)
    Settings.embed_model = resolve_embed_model(CONFIG["embedding_model"])
    print(f"✓ Embedding model configured: {CONFIG['embedding_model']}")

    # Set up node parser for chunking with hardcoded settings
    Settings.node_parser = SentenceSplitter(
        chunk_size=CONFIG["chunk_size"],
        chunk_overlap=CONFIG["chunk_overlap"]
    )
    print(f"✓ Text chunking configured: {CONFIG['chunk_size']} chars with {CONFIG['chunk_overlap']} overlap")

# Configure the settings
configure_llamaindex_settings()
print("✓ LlamaIndex settings configured for multimodal processing")

✓ LLM configured: gpt-5-mini


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✓ Embedding model configured: local:BAAI/bge-small-en-v1.5
✓ Text chunking configured: 512 chars with 50 overlap
✓ LlamaIndex settings configured for multimodal processing


## 5. Explore the dataset

Before loading documents, always inspect what files are present.

This helps you answer questions like:

- How many files are present?
- What file types are present?
- Are we working with PDFs only or multiple formats?
- Is the folder path correct?

In [17]:
def explore_dataset(data_path: str = None):
    """
    Explore and categorize the files in our dataset by type.

    Args:
        data_path (str): Path to the data directory
    """
    if data_path is None:
        data_path = CONFIG["data_path"]

    data_dir = Path(data_path)
    if not data_dir.exists():
        print(f"Data directory not found: {data_dir}")
        return

    # Categorize files by type
    file_types = {}
    all_files = []

    # Walk through all files recursively
    for file_path in data_dir.rglob("*"):
        if file_path.is_file():
            suffix = file_path.suffix.lower()
            file_size = file_path.stat().st_size

            if suffix not in file_types:
                file_types[suffix] = []

            file_info = {
                "path": str(file_path),
                "name": file_path.name,
                "size_mb": round(file_size / (1024 * 1024), 2),
                "size_bytes": file_size
            }

            file_types[suffix].append(file_info)
            all_files.append(file_info)

    # Display summary
    print("---Dataset Overview---")
    print(f"Total files found: {len(all_files)}")

    print(f"\nFile Types Distribution:")
    for file_type, files in sorted(file_types.items()):
        if file_type:  # Skip files without extension
            total_size = sum(f["size_mb"] for f in files)
            print(f"  {file_type}: {len(files)} files ({total_size:.2f} MB)")

            # Show file details
            for file_info in files[:3]:  # Show first 3 files of each type
                print(f"    - {file_info['name']} ({file_info['size_mb']} MB)")
            if len(files) > 3:
                print(f"    ... and {len(files) - 3} more")

            print()

    return file_types, all_files

# Explore our dataset
file_types, all_files = explore_dataset()
print(f"✓ Found {len(all_files)} files across {len(file_types)} different file types")


---Dataset Overview---
Total files found: 21

File Types Distribution:
  .csv: 4 files (0.00 MB)
    - agent_performance_benchmark.csv (0.0 MB)
    - agent_evaluation_metrics.csv (0.0 MB)
    - italian_recipes.csv (0.0 MB)
    ... and 1 more

  .html: 2 files (0.00 MB)
    - fitness_tracker.html (0.0 MB)
    - agent_tutorial.html (0.0 MB)

  .md: 4 files (0.00 MB)
    - agent_framework_comparison.md (0.0 MB)
    - recipe_instructions.md (0.0 MB)
    - city_guides.md (0.0 MB)
    ... and 1 more

  .mp3: 3 files (2.95 MB)
    - ai_agents.mp3 (1.54 MB)
    - rags.mp3 (0.81 MB)
    - in_the_end.mp3 (0.6 MB)

  .pdf: 2 files (1.92 MB)
    - Emerging_Agent_Architectures.pdf (1.58 MB)
    - AI_Agent_Frameworks.pdf (0.34 MB)

  .png: 6 files (0.55 MB)
    - agent_types_comparison.png (0.1 MB)
    - agent_performance_comparison.png (0.17 MB)
    - fitness_progress.png (0.08 MB)
    ... and 3 more

✓ Found 21 files across 6 different file types


## 6. Load multimodal documents

In the class notebook, we used `SimpleDirectoryReader` because it can load many file types such as PDF, CSV, Markdown, HTML, images, and notebooks.

At this stage, files become LlamaIndex `Document` objects.

### Your task

Complete the function below to load documents from the dataset folder.

In [15]:
from llama_index.core import SimpleDirectoryReader

def load_documents_from_folder(data_path: str):
    """
    Load documents from a folder using SimpleDirectoryReader.

    TODO: Complete this function to load documents from the given folder path.
    HINT: Use SimpleDirectoryReader with recursive parameter to load all files

    Args:
        data_path (str): Path to the folder containing data

    Returns:
        List of documents loaded from the folder
    """

    if data_path is None:
        data_path = CONFIG["data_path"]

    print(f"📂 Loading multimodal documents from: {data_path}")

    # Create SimpleDirectoryReader with recursive search
    reader = SimpleDirectoryReader(
        input_dir=data_path,
        recursive=True
        # Let SimpleDirectoryReader handle all supported file types automatically
    )

    print("🔄 Processing files...")
    start_time = time.time()

    # Load all documents
    documents = reader.load_data()

    end_time = time.time()

    print(f"✅ Successfully loaded {len(documents)} documents in {end_time - start_time:.2f} seconds")

    # Analyze loaded documents by file type
    doc_types = {}
    for doc in documents:
        file_type = doc.metadata.get('file_type', 'unknown')
        if file_type not in doc_types:
            doc_types[file_type] = []
        doc_types[file_type].append(doc)

    print(f"\n📊 Documents by MIME type:")
    for mime_type, docs in sorted(doc_types.items()):
        print(f"  {mime_type}: {len(docs)} documents")

    return documents

    # TODO: Create SimpleDirectoryReader (HINT: use recursive=True)
    # reader = ?

    # TODO: Load and return documents
    # documents = ?

    # return documents

# Test the function after you complete it
test_folder = CONFIG["data_path"]
documents = load_documents_from_folder(test_folder)
print(f"Loaded {len(documents)} documents")

📂 Loading multimodal documents from: /content/drive/MyDrive/ai-accelerator-C6-main/Day 6/session_2/data
🔄 Processing files...


100%|████████████████████████████████████████| 139M/139M [00:01<00:00, 125MiB/s]
/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


✅ Successfully loaded 42 documents in 61.32 seconds

📊 Documents by MIME type:
  application/pdf: 23 documents
  audio/mpeg: 3 documents
  image/png: 6 documents
  text/csv: 4 documents
  text/html: 2 documents
  text/markdown: 4 documents
Loaded 42 documents


## 7. Create LanceDB vector store

The vector store is where embeddings are stored and searched.

Important idea:

- LlamaIndex creates chunks and embeddings
- LanceDB stores the vector representation
- Later, the retriever searches this vector store

### Your task

Complete the function below.

In [44]:
import lancedb
# Vector store and index creation
from llama_index.vector_stores.lancedb import LanceDBVectorStore
from llama_index.core import StorageContext, VectorStoreIndex

def create_vector_store(vector_db_path: str, table_name: str):
    """
    Create a LanceDB vector store for storing document embeddings.

    TODO: In this function, you need to:
    1. Create the database directory if it does not already exist.
    2. Create a LanceDBVectorStore object.
    3. Return the vector store.

    Args:
        db_path (str): Path where the vector database will be stored.
        table_name (str): Name of the table in the vector database.

    Returns:
        LanceDBVectorStore: Configured vector store.
    """

    # TODO: Create the directory if it doesn't exist
    # Path(db_path).mkdir(parents=True, exist_ok=True)

    # TODO: Connect to LanceDB (creates a connection to the LanceDB database)
    #db = lancedb.connect(str(vector_db_path))

    # TODO: Create vector store (HINT: Use LanceDBVectorStore)
    # vector_store = ?

    # print("✓ LanceDB vector store created for multimodal data")

    # return vector_store
    #"""Create and configure LanceDB vector store for multimodal data."""
    if vector_db_path is None:
        vector_db_path = CONFIG["vector_db_path"]

    try:
        import lancedb

        # Create storage directory
        Path(vector_db_path).parent.mkdir(parents=True, exist_ok=True)

        # Connect to LanceDB
        db = lancedb.connect(str(vector_db_path))
        print(f"✓ Connected to LanceDB at: {vector_db_path}")

        # Create vector store
        vector_store = LanceDBVectorStore(
            uri=str(vector_db_path),
            table_name="multimodal_documents",
            show_progress=True
        )
        print("✓ LanceDB vector store created for multimodal data")

        return vector_store

    except Exception as e:
        print(f"Error creating vector store: {e}")
        return None

# Test the function after you complete it
vector_store = create_vector_store(vector_db_path=CONFIG["vector_db_path"], table_name=CONFIG["table_name"])
print(f"Vector store created: {vector_store is not None}")

✓ Connected to LanceDB at: /content/storage/assignment_multimodal_vectordb
✓ LanceDB vector store created for multimodal data
Vector store created: True


## 8. Create StorageContext and VectorStoreIndex

This is one of the most important parts of the assignment.

### What is `StorageContext`?

Think of it as the object that tells LlamaIndex where the prepared index data should live.

In this assignment:

- `vector_store` stores embeddings
- `StorageContext` connects LlamaIndex to that vector store
- `VectorStoreIndex.from_documents()` builds the index from documents

### Your task

Complete the function below.

In [45]:
def create_vector_index(documents: List, vector_store, persist_dir: str = None):
    """
    Create a VectorStoreIndex from loaded documents.

    Args:
        documents: Loaded LlamaIndex documents.
        vector_store: LanceDB vector store.
        persist_dir: Folder for persisting index metadata.

    Returns:
        VectorStoreIndex object.
    """
    if persist_dir is None:
        persist_dir = CONFIG["index_storage_path"]

    if not documents:
        raise ValueError("No documents found. Load documents before creating the index.")

    Path(persist_dir).mkdir(parents=True, exist_ok=True)

    print("Creating storage context")
    # TODO: Create storage context from vector_store (HINT: Use StorageContext)
    # storage_context = ?

    # TODO: Create the VectorStoreIndex from documents
    # index = ?

    # TODO: Persist index metadata
    # index.storage_context.persist(persist_dir=persist_dir)

    # return index
    #"""Create or load a multimodal vector index."""

    if index_storage_path is None:
        index_storage_path = CONFIG["index_storage_path"]

    index_path = Path(index_storage_path)
    index_path.mkdir(parents=True, exist_ok=True)

    # Check if index already exists
    index_store_file = index_path / "index_store.json"

    if not force_rebuild and index_store_file.exists():
        print("📁 Loading existing multimodal index...")
        try:
            storage_context = StorageContext.from_defaults(
                persist_dir=str(index_path),
                vector_store=vector_store
            )

            index = VectorStoreIndex.from_vector_store(
                vector_store=vector_store,
                storage_context=storage_context
            )
            print("✓ Successfully loaded existing multimodal index")
            return index

        except Exception as e:
            print(f"⚠️  Error loading existing index: {e}")
            print("Creating new index...")

    if not documents:
        print("❌ No documents to index")
        return None

    print("🔨 Creating new multimodal vector index...")
    start_time = time.time()

    # Create storage context with vector store
    storage_context = StorageContext.from_defaults(vector_store=vector_store)

    # Create index with progress bar
    index = VectorStoreIndex.from_documents(
        documents,
        storage_context=storage_context,
        show_progress=True
    )

    end_time = time.time()
    print(f"✓ Multimodal index created in {end_time - start_time:.2f} seconds")

    # Save index to storage
    print("💾 Saving multimodal index to storage...")
    index.storage_context.persist(persist_dir=str(index_path))
    print("✓ Index saved successfully")

    return index

if vector_store and documents:
    index = create_vector_index(documents, vector_store)

## 9. Create a retriever
A retriever does not generate an answer. It only returns the most relevant chunks.

This is useful because it lets you inspect what the RAG system found before the LLM answers.

### Your task

Complete the retriever setup.

In [ ]:
from llama_index.core.retrievers import VectorIndexRetriever

def create_retriever(index, similarity_top_k: int = None):
    """
    Create a retriever from the index.

    Args:
        index: VectorStoreIndex object.
        similarity_top_k: Number of chunks to retrieve.

    Returns:
        VectorIndexRetriever object.
    """
    if similarity_top_k is None:
        similarity_top_k = CONFIG["similarity_top_k"]

    # TODO: Create VectorIndexRetriever
    # retriever = ?

    # print(f"Retriever created with similarity_top_k={similarity_top_k}")
    # return retriever

retriever = create_retriever(index)

## 10. Retrieve chunks for a query

This step helps you see exactly what semantic search returns.

Good queries to try:

- Ask about a topic you know exists in the dataset
- Ask using different wording than the original document
- Ask a broad question and inspect whether results are noisy

In [ ]:
def retrieve_chunks(retriever, query: str, show_text: bool = True):
    """
    Retrieve relevant chunks for a query and print their metadata and score.

    Args:
        retriever: VectorIndexRetriever object.
        query: User query.
        show_text: Whether to print text previews.

    Returns:
        Retrieved nodes.
    """
    print("Query:", query)
    print("=" * 80)

    nodes = retriever.retrieve(query)

    print("Retrieved chunks:", len(nodes))

    for i, node_with_score in enumerate(nodes, 1):
        node = node_with_score.node
        score = node_with_score.score
        metadata = node.metadata

        print(f"Result {i}")
        print("Score:", score)
        print("File name:", metadata.get("file_name", "unknown"))
        print("File type:", metadata.get("file_type", "unknown"))

        if show_text:
            print("Text preview:")
            print(node.get_content()[:700].replace("", " "))

        print("-" * 80)

    return nodes

sample_query = "Where should I travel next in May-June?"
retrieved_nodes = retrieve_chunks(retriever, sample_query)

## 11. Build a query engine

A retriever only returns chunks. A query engine uses the retriever and an LLM to generate a final response.

This mirrors the class notebook section where we created a multimodal query engine.

### Your task

Complete the function below.

In [ ]:
from llama_index.core.query_engine import RetrieverQueryEngine

def create_query_engine(retriever, similarity_top_k: int = None):
    """
    Create a RetrieverQueryEngine using a retriever.

    Args:
        retriever: VectorIndexRetriever object.
        similarity_top_k: Number of chunks to retrieve.

    Returns:
        RetrieverQueryEngine object.
    """
    if similarity_top_k is None:
        similarity_top_k = CONFIG["similarity_top_k"]

    # TODO: Create query engine (HINT: Use RetrieverQueryEngine)
    # query_engine = ?

    # print("Query engine created")
    # return query_engine

query_engine = create_query_engine(retriever)

## 12. Ask a question and inspect sources

This is the final RAG step.

We ask a question, get a generated answer, and then inspect which source chunks were used.

In [ ]:
def ask_question(query_engine, question: str, show_sources: bool = True):
    """
    Ask a question to the query engine and display answer plus sources.

    Args:
        query_engine: RetrieverQueryEngine object.
        question: User question.
        show_sources: Whether to show source chunks.
    """
    print("Question:", question)
    print("=" * 80)

    response = query_engine.query(question)

    print("Answer:")
    print(str(response))

    if show_sources:
        print("Sources used:")
        source_nodes = getattr(response, "source_nodes", [])
        for i, source in enumerate(source_nodes, 1):
            node = source.node
            metadata = node.metadata
            print(f"Source {i}")
            print("Score:", source.score)
            print("File name:", metadata.get("file_name", "unknown"))
            print("File type:", metadata.get("file_type", "unknown"))
            print("Text preview:")
            print(node.get_content()[:500].replace("", " "))
            print("-" * 80)

# Try your own question here
question = "What are the steps to make Carbonara?"
ask_question(query_engine, question, show_sources=True)

## Conclusion

🎉 **Congratulations!** You have successfully built an advanced **Multimodal RAG System** using LlamaIndex's `SimpleDirectoryReader` with comprehensive cross-modal capabilities.

## Student experiments (for trying out later after the session)

Complete these small experiments later to build intuition.

### Experiment 1: Change `similarity_top_k`

Try values like 2, 5, and 10.

Question to answer:

- Does increasing `top_k` always improve the answer?
- Do you see more noise when `top_k` is high?

### Experiment 2: Ask the same question in different wording

Example:

- "What is the refund policy?"
- "Can customers get their money back?"

Question to answer:

- Does semantic search still retrieve similar chunks?

### Experiment 3: Inspect sources before trusting the answer

Question to answer:

- Did the LLM answer using the right sources?
- Were any irrelevant chunks included?

In [ ]:
# Experiment area
# Change the query and top_k values below.

experiment_query = "Replace this with your own question"
experiment_top_k = 3

experiment_retriever = create_retriever(index, similarity_top_k=experiment_top_k)
experiment_nodes = retrieve_chunks(experiment_retriever, experiment_query)

## Reflection questions

Answer these in a markdown cell below after you complete the notebook.

1. Why do we parse and load files before indexing?
2. Why do we chunk text instead of storing the full document as one unit?
3. What does `chunk_overlap` help with?
4. What is stored in the vector database?
5. What role does `StorageContext` play?
6. What is the difference between retriever output and query engine output?
7. Give one example where returning retrieved chunks directly is better than generating an answer.
8. Give one example where generation is useful after retrieval.

In [ ]:
# Write short answers here as comments or create a markdown cell below.

# 1.
# 2.
# 3.
# 4.
# 5.
# 6.
# 7.
# 8.